# Практика: Деревья решений, ансамбли и Bias-Variance Decomposition

**Подготовка к Всероссийской олимпиаде по ИИ, Сириус 2026**

---

### Содержание

1. **Часть 1.** Решающие деревья — теория и реализация с нуля
2. **Часть 2.** Деревья в sklearn: границы решений и визуализация
3. **Часть 3.** Bias-Variance Decomposition — математика и эксперименты
4. **Часть 4.** Бэггинг: от идеи к реализации
5. **Часть 5.** Случайный лес: теория и практика
6. **Часть 6.** Задачи на реальных данных: предсказание оттока абонентов
7. **Часть 7.** Итоговые задания и вопросы на интерпретацию

---

<img src="https://64.media.tumblr.com/5094cb9cbb5687c23a68f00d41fc8c5c/tumblr_nuwi9wAy071udkhfio1_1280.jpg" width="500">

In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.datasets import load_iris, make_moons, load_breast_cancer, load_digits
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, BaggingRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn import tree

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 13

---

## Часть 1. Решающие деревья — теория и реализация с нуля

### 1.1. Формальная постановка

Дана обучающая выборка $X = \{(x_i, y_i)\}_{i=1}^{N}$, где $x_i \in \mathbb{R}^d$, $y_i \in \{1, \ldots, K\}$ (задача классификации).

**Решающее дерево** — это функция $a(x): \mathbb{R}^d \to \{1, \ldots, K\}$, задаваемая бинарным деревом, в каждом внутреннем узле которого находится предикат вида $[x^{(j)} \leqslant t]$, а в листьях — метки классов.

### 1.2. Критерий информативности (Impurity)

Пусть $X_m$ — множество объектов, попавших в узел $m$. Обозначим долю объектов класса $k$:

$$
p_k = \frac{1}{|X_m|} \sum_{i: x_i \in X_m} [y_i = k]
$$

**Энтропия Шеннона:**

$$
H(X_m) = -\sum_{k=1}^{K} p_k \log_2 p_k
$$

Свойства энтропии:
- $H = 0$, если все объекты одного класса (идеальная чистота)
- $H = \log_2 K$, если все классы равновероятны (максимальная неопределённость)
- Для бинарной классификации: $H(p) = -p\log_2 p - (1-p)\log_2(1-p)$, максимум при $p = 0.5$

### 1.3. Information Gain

Для разбиения по признаку $j$ с порогом $t$:
- $X_l = \{x_i \in X_m : x_i^{(j)} \leqslant t\}$, $X_r = \{x_i \in X_m : x_i^{(j)} > t\}$

$$
\text{IG}(X_m, j, t) = H(X_m) - \frac{|X_l|}{|X_m|} H(X_l) - \frac{|X_r|}{|X_m|} H(X_r) \to \max_{j, t}
$$

Заметим, что $\frac{|X_l|}{|X_m|} H(X_l) + \frac{|X_r|}{|X_m|} H(X_r)$ — это условная энтропия $H(Y | X^{(j)} \leqslant t)$, а Information Gain — взаимная информация $I(Y; [X^{(j)} \leqslant t])$.

### 1.4. Критерий Джини

**Индекс Джини:**

$$
G(X_m) = 1 - \sum_{k=1}^{K} p_k^2 = \sum_{k=1}^{K} p_k(1 - p_k)
$$

**Интерпретация:** $G(X_m)$ — вероятность ошибки случайного классификатора, который предсказывает класс $k$ с вероятностью $p_k$.

**Связь с энтропией:** разложим $-\ln p$ в ряд Тейлора около $p=1$:

$$
-\ln p = (1-p) + \frac{(1-p)^2}{2} + \ldots \approx (1-p)
$$

Тогда $H \approx \sum_k p_k(1-p_k) = G$. Джини — это аппроксимация энтропии первого порядка.

### 1.5. Misclassification Rate

Ещё один критерий (хуже на практике):

$$
\text{Err}(X_m) = 1 - \max_k p_k
$$

### 1.6. Для задачи регрессии

Пусть $y_i \in \mathbb{R}$. Критерий качества узла — **дисперсия** (MSE):

$$
D(X_m) = \frac{1}{|X_m|} \sum_{i: x_i \in X_m} (y_i - \bar{y}_m)^2, \qquad \bar{y}_m = \frac{1}{|X_m|} \sum_{i: x_i \in X_m} y_i
$$

В листе предсказание — среднее: $\hat{y} = \bar{y}_m = \arg\min_c \sum_{i \in X_m} (y_i - c)^2$.

Для MAE (абсолютная ошибка) предсказание — медиана.

### 1.7. Жадный алгоритм построения

Сложность полного перебора **всех** деревьев экспоненциальна. Поэтому деревья строятся **жадно**:

$$
(j^*, t^*) = \arg\max_{j \in \{1,\ldots,d\},\; t \in \mathcal{T}_j} \text{IG}(X_m, j, t)
$$

где $\mathcal{T}_j$ — множество уникальных пороговых значений признака $j$ (или средних между соседними значениями).

Сложность одного разбиения: $O(d \cdot N \log N)$ (сортировка по каждому признаку).

### 1.8. Критерии останова

- Максимальная глубина дерева (`max_depth`)
- Минимальное число объектов в узле (`min_samples_split`)
- Минимальное число объектов в листе (`min_samples_leaf`)
- Все объекты в узле принадлежат одному классу ($H = 0$)
- Information Gain < порога
- Максимальное число листьев (`max_leaf_nodes`)

### Задание 1.1. Реализация дерева решений с нуля

Реализуйте класс `Node` — узел дерева решений. Каждый узел хранит:
- Левое и правое поддерево
- Признак и порог разбиения
- Метку класса (если лист)

In [ ]:
class Node:
    def __init__(self, max_depth=None, depth=0):
        self.left = None
        self.right = None
        self.feature = None
        self.threshold = None
        self.label = None
        self.max_depth = max_depth
        self.depth = depth

    def H(self, y: np.array) -> float:
        """Вычислите энтропию Шеннона для массива меток y."""
        y_dict = dict()

        for i in y:
            if y_dict.get(y) is None:
                y_dict[y] = 0
            y_dict[y] += 1

        H = 0

        for i in set(y):
            p_i = y_dict[i] / len(y)
            H += p_i * np.log(1 / p_i)
    
        return -H

    def Q(self, y: np.array, y_left: np.array, y_right: np.array) -> float:
        """Вычислите Information Gain при разбиении y на y_left и y_right."""
        H_l = self.H(y_left)
        H_r = self.H(y_right)
        H_m = self.H(y)

        return H_m - abs(H_l / H_m) * H_l - abs(H_r / H_m) * H_r

    def is_stop(self, y: np.array) -> bool:
        """Проверьте критерий останова: все метки одинаковы или достигнута max_depth."""
        if self.depth >= self.max_depth:
            return False
        
        if len(np.unique(y)) <= 1:
            return False
        
        return True

    def fit(self, X: np.array, y: np.array):
        """Рекурсивное построение дерева.
        Для каждого признака и каждого уникального порога:
        1. Разбить данные на левую и правую части
        2. Вычислить Information Gain
        3. Выбрать лучшее разбиение
        4. Рекурсивно построить поддеревья
        """

        best_split = {
            'IG': -np.inf,
            'f_ind': 0,
            'treshold': 0
        }

        for f_i in range(X.shape[1]):
            x_th = np.sort(np.unique(X[:, f_i]))

            for cur_th in x_th:
                l_mask = (X <= cur_th)
                y_l = y[l_mask]
                y_r = y[~l_mask]

                cur_IG = self.Q(y, y_l, y_r)
                if cur_IG > best_split['IG']:
                    best_split['IG'] = cur_IG
                    best_split['f_ind'] = f_i
                    best_split['treshold'] = cur_th


        

### Задание 1.2. Класс `DecisionTree`

Реализуйте обёртку над `Node` с методами `fit`, `predict` и `print_conditions`.

In [ ]:
class DecisionTree:
    def __init__(self, max_depth=None):
        self.root = None
        self.max_depth = max_depth

    def fit(self, X, y):
        # YOUR CODE HERE
        pass

    def predict_one(self, node, x):
        """Предсказание для одного объекта: рекурсивный обход дерева."""
        # YOUR CODE HERE
        pass

    def predict(self, X):
        # YOUR CODE HERE
        pass

    def print_conditions(self, node=None, indent=''):
        """Рекурсивная печать правил дерева."""
        # YOUR CODE HERE
        pass

### Задание 1.3. Тестирование на Iris

1. Загрузите Iris (бинарная задача: `target < 2`)
2. Обучите **ваше** дерево
3. Выведите accuracy
4. Напечатайте правила дерева через `print_conditions`
5. Сравните accuracy с `DecisionTreeClassifier` из sklearn

In [ ]:
iris = load_iris()
mask = iris.target < 2
X_iris, y_iris = iris.data[mask], iris.target[mask]

# YOUR CODE HERE

### Задание 1.4. Дерево для регрессии (бонус)

Модифицируйте ваш `Node`, чтобы поддерживать регрессию (MSE вместо энтропии, среднее вместо мажоритарного класса). Протестируйте на `sin(x)` с шумом.

In [ ]:
# YOUR CODE HERE

**Вопрос 1.1.** Вычислите руками энтропию и индекс Джини для узла с распределением классов $(0.1, 0.2, 0.7)$. Какой критерий «сильнее» штрафует неравномерность?

*Ваш ответ:*

**Вопрос 1.2.** Почему жадный алгоритм может построить неоптимальное дерево? Приведите пример данных (можно нарисовать), где оптимальное дерево глубины 2 не содержит оптимальный первый сплит.

*Ваш ответ:*

---

## Часть 2. Деревья в sklearn: границы решений и визуализация

### Немного нового полезного из работы с матрицами:
- запустите ячейки с примерами функций
- посмотрите на получаемый результат, проинтерпретируйте его

In [ ]:
# meshgrid — создаёт координатную сетку из двух векторов
a = [1, 2, 3, 4, 5]
b = [0, 5, 10, 15, 20, 25, 30]
AB = np.meshgrid(a, b)
print("X-координаты:")
print(AB[0])
print("\nY-координаты:")
print(AB[1])

In [ ]:
# ravel — развернуть матрицу в вектор
a = np.array([[0, 1, 2],
               [3, 4, 5],
               [6, 7, 8]])
print(np.ravel(a))

In [ ]:
# np.c_ — конкатенация столбцов
a = np.arange(1, 5, 1)
b = np.arange(10, 41, 10)
d = np.arange(100, 401, 100)
print(np.c_[a, b, d])

In [ ]:
# Все пары — meshgrid + ravel + np.c_
a = [1, 2, 3, 4]
b = [0, 10, 20, 30]
AB = np.meshgrid(a, b)
print(np.c_[AB[0].ravel(), AB[1].ravel()])

In [ ]:
# pcolormesh — визуализация матрицы цветом
c = [[ 0, 1, 3, 4],
     [ 4, 5, 6, 7],
     [ 8, 9, 10, 11],
     [12, 13, 14, 15]]

plt.pcolormesh(c, edgecolor='w', linewidth=1)
plt.colorbar()
plt.title('pcolormesh: визуализация матрицы')
plt.show()

### Задание 2.1. Функция визуализации границ решений

Напишите функцию `plot_decision_boundary(clf, X, y, title)`, которая:
1. Создаёт сетку точек с помощью `np.meshgrid` и `np.linspace`
2. Предсказывает класс для каждой точки сетки
3. Рисует раскраску через `plt.pcolormesh` и точки данных через `plt.scatter`

In [ ]:
def plot_decision_boundary(clf, X, y, title=''):
    """Визуализация границ решений классификатора."""
    # YOUR CODE HERE
    pass

### Задание 2.2. Границы решений на make_moons

1. Сгенерируйте данные `make_moons(n_samples=500, noise=0.25, random_state=42)`
2. Визуализируйте сами данные (scatter plot)
3. Обучите деревья с `criterion='entropy'` и глубинами 1, 2, 5, 9
4. Для каждого постройте границу решений (subplots 2×2)
5. В заголовок добавьте train accuracy

In [ ]:
X, y = make_moons(n_samples=500, noise=0.25, random_state=42)

# YOUR CODE HERE

### Задание 2.3. Визуализация структуры дерева

Обучите дерево глубины 3 и визуализируйте его с помощью `tree.plot_tree`. Используйте параметры `filled=True, rounded=True`.

In [ ]:
# YOUR CODE HERE

**Вопрос 2.1.** Как изменяется граница решений при увеличении глубины дерева? В какой момент наступает переобучение? Как вы это определили?

*Ваш ответ:*

### Задание 2.4. Кросс-валидация и выбор глубины

Постройте на одном графике **train accuracy** и **CV accuracy** (5-fold) в зависимости от `max_depth` (от 1 до 20). Используйте `cross_val_score`.

In [ ]:
# YOUR CODE HERE

**Вопрос 2.2.** При какой глубине CV accuracy максимальна? Почему train accuracy растёт монотонно, а CV — нет?

*Ваш ответ:*

**Вопрос 2.3.** Дерево решений разбивает пространство признаков на прямоугольные области (параллельные осям). Какие данные будут плохо аппроксимироваться деревом? Приведите пример.

*Ваш ответ:*

### Задание 2.5. Gini vs Entropy

Обучите два дерева одинаковой глубины (5) с `criterion='gini'` и `criterion='entropy'`. Сравните CV accuracy. Визуализируйте границы решений. Есть ли значимая разница?

In [ ]:
# YOUR CODE HERE

---

## Часть 3. Bias-Variance Decomposition

### 3.1. Постановка

Одна из ключевых концепций ML — **разложение ошибки на смещение и разброс**.

Пусть истинная зависимость: $y = f(x) + \varepsilon$, где $\varepsilon$ — шум, $\mathbb{E}[\varepsilon] = 0$, $\text{Var}(\varepsilon) = \sigma^2$.

Пусть $\hat{f}(x)$ — модель, обученная на выборке $D \sim P^N$. Рассмотрим **ожидаемую ошибку** в точке $x$:

$$
\text{Err}(x) = \mathbb{E}_{D, \varepsilon}\big[(y - \hat{f}(x))^2\big]
$$

### 3.2. Доказательство разложения

Обозначим $\bar{f}(x) = \mathbb{E}_D[\hat{f}(x)]$ — усреднённое по всем возможным выборкам предсказание.

$$
\text{Err}(x) = \mathbb{E}_{D,\varepsilon}\big[(f(x) + \varepsilon - \hat{f}(x))^2\big]
$$

Раскроем квадрат и воспользуемся независимостью $\varepsilon$ от $\hat{f}$:

$$
= \mathbb{E}_D\big[(f(x) - \hat{f}(x))^2\big] + 2\,\underbrace{\mathbb{E}_{D,\varepsilon}\big[(f(x) - \hat{f}(x))\varepsilon\big]}_{=0} + \mathbb{E}[\varepsilon^2]
$$

$$
= \mathbb{E}_D\big[(f(x) - \hat{f}(x))^2\big] + \sigma^2
$$

Прибавим и вычтем $\bar{f}(x)$:

$$
\mathbb{E}_D\big[(f - \hat{f})^2\big] = \mathbb{E}_D\big[(f - \bar{f} + \bar{f} - \hat{f})^2\big]
$$

$$
= (f - \bar{f})^2 + 2(f - \bar{f})\underbrace{\mathbb{E}_D[\bar{f} - \hat{f}]}_{=0} + \mathbb{E}_D[(\bar{f} - \hat{f})^2]
$$

**Итого:**

$$
\boxed{\text{Err}(x) = \underbrace{\big(f(x) - \bar{f}(x)\big)^2}_{\text{Bias}^2(x)} + \underbrace{\mathbb{E}_D\big[(\hat{f}(x) - \bar{f}(x))^2\big]}_{\text{Variance}(x)} + \underbrace{\sigma^2}_{\text{Noise}}}
$$

### 3.3. Интерпретация

| | Высокий Bias | Низкий Bias |
|---|---|---|
| **Высокий Variance** | Плохая модель | Переобучение (overfitting) |
| **Низкий Variance** | Недообучение (underfitting) | Идеальная модель |

- **Bias** — систематическая ошибка: насколько усреднённое предсказание отличается от истины
- **Variance** — чувствительность к обучающей выборке: насколько предсказание "прыгает" на разных $D$
- **Noise** — неустранимая ошибка: стохастичность самих данных

### 3.4. Связь со сложностью модели

| Модель | Bias | Variance | Пример |
|--------|------|----------|--------|
| Константа | Высокий | 0 | $\hat{f}(x) = \bar{y}$ |
| Линейная регрессия | Средний | Низкий | $\hat{f}(x) = w^T x$ |
| Неглубокое дерево | Средний | Низкий | `max_depth=3` |
| Глубокое дерево | Низкий | Высокий | `max_depth=None` |
| KNN ($k$ большое) | Высокий | Низкий | $k = N$ |
| KNN ($k=1$) | Низкий | Высокий | |

### 3.5. Роль ансамблей

- **Бэггинг / Random Forest:** усреднение снижает **Variance**, Bias остаётся примерно таким же
- **Бустинг:** последовательно снижает **Bias** (каждая новая модель исправляет ошибки предыдущих)

### Задание 3.1. Экспериментальная проверка B-V Decomposition

Проведите Монте-Карло эксперимент:
1. Зафиксируйте истинную функцию $f(x) = \sin(2\pi x)$ и параметры: `n_experiments=200`, `n_train=50`, `noise_std=0.3`
2. Создайте тестовую сетку `x_test = np.linspace(0, 1, 100)`
3. Для каждой глубины из `[1, 2, 3, 5, 8, 12, 20]`:
   - 200 раз сгенерируйте выборку, обучите `DecisionTreeRegressor`, сохраните предсказания на `x_test`
   - Вычислите `Bias² = mean((f_true - mean_predictions)²)` и `Variance = mean(var(predictions))`
4. Постройте графики Bias², Variance, MSE и горизонтальную линию Noise ($\sigma^2$) на одном графике
5. Проверьте, что $\text{MSE} \approx \text{Bias}^2 + \text{Variance} + \sigma^2$

In [ ]:
np.random.seed(42)

def true_function(x):
    return np.sin(2 * np.pi * x)

n_experiments = 200
n_train = 50
noise_std = 0.3

# YOUR CODE HERE

### Задание 3.2. Визуализация предсказаний для разных глубин

Для глубин `[1, 3, 8, 20]` постройте subplots 2×2:
- На каждом — 20 полупрозрачных линий (предсказания на разных выборках)
- Жирная красная линия — среднее предсказание $\bar{f}(x)$
- Чёрная пунктирная — истинная функция $f(x)$

In [ ]:
# YOUR CODE HERE

**Вопрос 3.1.** Объясните наблюдаемую картину. Почему при `depth=1` синие линии кучкуются, но далеки от чёрной? Почему при `depth=20` — наоборот?

*Ваш ответ:*

**Вопрос 3.2.** Выведите таблицу с колонками: Depth, Bias², Variance, Bias²+Var+σ², MSE. Совпадают ли последние два столбца?

*Ваш ответ:*

In [ ]:
# YOUR CODE HERE

### Задание 3.3. B-V Decomposition для других моделей

Повторите эксперимент из 3.1, но вместо деревьев разной глубины сравните:
- KNN с $k \in \{1, 3, 10, 30, 50\}$

Как меняются Bias и Variance при увеличении $k$? Согласуется ли с теорией?

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# YOUR CODE HERE

---

## Часть 4. Бэггинг: от идеи к реализации

### 4.1. Теоретическое обоснование

**Бэггинг** (Bootstrap Aggregating, Breiman, 1996) — метод снижения дисперсии ансамбля.

**Пусть** $\hat{f}_1, \ldots, \hat{f}_B$ — модели, обученные на разных подвыборках. Рассмотрим их среднее:

$$
\hat{f}_{\text{bag}}(x) = \frac{1}{B} \sum_{b=1}^{B} \hat{f}_b(x)
$$

**Случай независимых моделей** (одинаковая дисперсия $\sigma^2$):

$$
\text{Var}(\hat{f}_{\text{bag}}) = \frac{\sigma^2}{B} \xrightarrow{B \to \infty} 0
$$

**Реальный случай** — модели коррелированы (обучены на пересекающихся выборках). Пусть $\text{Corr}(\hat{f}_b, \hat{f}_{b'}) = \rho$ для $b \neq b'$:

$$
\text{Var}(\hat{f}_{\text{bag}}) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2
$$

**Доказательство:**

$$
\text{Var}\!\left(\frac{1}{B}\sum_b \hat{f}_b\right) = \frac{1}{B^2}\left[\sum_b \text{Var}(\hat{f}_b) + \sum_{b \neq b'} \text{Cov}(\hat{f}_b, \hat{f}_{b'})\right]
$$

$$
= \frac{1}{B^2}\left[B\sigma^2 + B(B-1)\rho\sigma^2\right] = \frac{\sigma^2}{B} + \frac{B-1}{B}\rho\sigma^2
$$

При $B \to \infty$ остаётся $\rho\sigma^2$ — дисперсия **не стремится к нулю** из-за корреляции.

### 4.2. Бутстрап

Генерируем $B$ подвыборок размера $N$ **с возвращением** из исходной выборки.

Вероятность того, что $i$-й объект **не попадёт** в бутстрап-выборку:

$$
P(\text{объект не выбран}) = \left(1 - \frac{1}{N}\right)^N \xrightarrow{N \to \infty} e^{-1} \approx 0.368
$$

Значит, ~63.2% объектов попадут в подвыборку, ~36.8% — **out-of-bag (OOB)**.

### 4.3. Влияние на Bias

Бэггинг **не меняет** Bias (в среднем): $\mathbb{E}[\hat{f}_{\text{bag}}] = \frac{1}{B}\sum \mathbb{E}[\hat{f}_b] = \mathbb{E}[\hat{f}]$.

### Задание 4.1. Реализация бэггинга с нуля

Реализуйте `BaggingDecisionClassifier` с `fit` (бутстрап + обучение деревьев) и `predict` (мажоритарное голосование).

In [ ]:
class BaggingDecisionClassifier:
    def __init__(self, n_estimators=10, max_depth=None, sample_size=1.0):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.sample_size = sample_size
        self.estimators = []

    def fit(self, X, y):
        # YOUR CODE HERE
        pass

    def predict(self, X):
        # YOUR CODE HERE
        pass

### Задание 4.2. Тестирование бэггинга

На `breast_cancer`:
1. Посчитайте 5-fold CV accuracy одного дерева (`max_depth=5`)
2. Посчитайте accuracy вашего `BaggingDecisionClassifier` (`n_estimators=50, max_depth=5`)
3. Сравните с `BaggingClassifier` из sklearn
4. Выведите результаты таблицей

In [ ]:
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target

# YOUR CODE HERE

### Задание 4.3. Bias-Variance: дерево vs бэггинг

Повторите Монте-Карло из Части 3, сравнив:
- Одиночное дерево (`max_depth=8`)
- `BaggingRegressor` из sklearn (50 деревьев, `max_depth=8`)

Визуализируйте предсказания (полупрозрачные линии + среднее + $f(x)$) для обоих. Вычислите Bias² и Variance.

In [ ]:
# YOUR CODE HERE

**Вопрос 4.1.** Как бэггинг повлиял на Bias и Variance? Согласуется ли с формулой $\text{Var} = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$?

*Ваш ответ:*

**Вопрос 4.2.** Если базовый алгоритм — линейная регрессия, поможет ли бэггинг? Почему?

*Ваш ответ:*

### Задание 4.4 (повышенной сложности). OOB Error

Реализуйте подсчёт **OOB-ошибки**: для каждого объекта $x_i$ предсказание делается только теми деревьями, в чьи бутстрап-выборки $x_i$ **не** вошёл.

Сравните OOB-error с 5-fold CV error.

In [ ]:
# YOUR CODE HERE

---

## Часть 5. Случайный лес (Random Forest)

### 5.1. Мотивация

В бэггинге деревья **коррелированы** (особенно если есть сильный признак — все деревья будут его использовать первым). Из формулы:

$$
\text{Var}(\hat{f}_{\text{bag}}) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2
$$

видно, что для дальнейшего снижения Variance нужно уменьшить $\rho$.

### 5.2. Идея Random Forest (Breiman, 2001)

При каждом разбиении узла рассматриваем не все $d$ признаков, а **случайное подмножество** из $m$ признаков. Это **декоррелирует** деревья.

**Типичные значения:**
- Классификация: $m = \lfloor\sqrt{d}\rfloor$
- Регрессия: $m = \lfloor d/3 \rfloor$

### 5.3. Алгоритм

Для $b = 1, \ldots, B$:
1. Сгенерировать бутстрап-выборку $D_b$ размера $N$
2. Построить дерево $T_b$ на $D_b$. В каждом узле:
   - Выбрать случайные $m$ признаков из $d$
   - Найти лучшее разбиение **среди этих $m$ признаков**
3. Не выполнять pruning (обычно деревья максимально глубокие)

Предсказание: $\hat{f}_{\text{RF}}(x) = \frac{1}{B}\sum_{b=1}^B T_b(x)$ (регрессия) или мажоритарное голосование (классификация).

### 5.4. Feature Importance

**MDI (Mean Decrease in Impurity):** для признака $j$ — суммарное уменьшение impurity по всем узлам всех деревьев, где использован признак $j$.

$$
\text{Imp}(j) = \frac{1}{B}\sum_{b=1}^{B}\sum_{t \in T_b: \text{split on } j} \frac{|X_t|}{N}\,\Delta G(t)
$$

**Проблема MDI:** завышает важность признаков с большим числом уникальных значений.

**Permutation Importance:** перемешиваем значения признака $j$ и смотрим, насколько ухудшилось качество.

### Задание 5.1. Реализация Random Forest с нуля

Реализуйте класс `MyRandomForest`. Ключевое отличие от бэггинга: при обучении каждого дерева используйте случайное подмножество признаков.

In [ ]:
class MyRandomForest:
    def __init__(self, n_estimators=100, max_depth=None, max_features='sqrt'):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.estimators = []
        self.feature_indices = []

    def fit(self, X, y):
        # YOUR CODE HERE
        pass

    def predict(self, X):
        # YOUR CODE HERE
        pass

### Задание 5.2. Сравнение на digits

1. Загрузите `digits`, визуализируйте несколько примеров
2. Сравните (5-fold CV accuracy):
   - KNN (подберите лучший `k` из [1, 3, 5, 7])
   - Одиночное дерево
   - Ваш `BaggingDecisionClassifier`
   - Ваш `MyRandomForest`
   - sklearn `RandomForestClassifier`
3. Выведите результаты в `pd.DataFrame`

In [ ]:
digits = load_digits()
X_d, y_d = digits.data, digits.target

# YOUR CODE HERE

### Задание 5.3. Исследование гиперпараметров

Постройте **три отдельных графика** CV accuracy от:
1. `n_estimators` (1–200, шаг 10)
2. `max_features` (1–$d$, шаг 2)
3. `max_depth` (1–30) — **два графика на одной оси**: `max_features='sqrt'` и `max_features=d`

Для п.2 добавьте вертикальную линию в $\sqrt{d}$.

In [ ]:
# YOUR CODE HERE

**Вопрос 5.1.** Почему `max_features='sqrt'` часто лучше `max_features=d`? Объясните через Bias-Variance.

*Ваш ответ:*

**Вопрос 5.2.** При каком `n_estimators` качество стабилизируется? Почему увеличение числа деревьев **не приводит к переобучению** (в отличие от бустинга)?

*Ваш ответ:*

**Вопрос 5.3.** Сложность обучения Random Forest: $O(B \cdot m \cdot N \log^2 N)$. Выведите эту оценку из сложности обучения одного дерева.

*Ваш ответ:*

### Задание 5.4. GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'max_features': ['sqrt', 'log2', 10],
    'min_samples_leaf': [1, 3, 5]
}

# YOUR CODE HERE: GridSearchCV → лучшие параметры и score

### Задание 5.5. Анализ ошибок

1. Обучите лучший RF на train, предскажите на test
2. Найдите все ошибочно классифицированные цифры
3. Визуализируйте их (imshow, 8×8) с подписями «True: X, Pred: Y»

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_d, y_d, test_size=0.2, random_state=42)

# YOUR CODE HERE

### Задание 5.6 (повышенной сложности). Permutation Importance

Реализуйте **permutation importance** с нуля:
1. Обучите RF, вычислите baseline accuracy на test
2. Для каждого признака $j$: перемешайте столбец $j$ в `X_test`, вычислите accuracy, верните столбец назад
3. Importance = baseline_acc - permuted_acc

Визуализируйте top-20 признаков. Сравните с `feature_importances_` из sklearn.

In [ ]:
# YOUR CODE HERE

---

## Часть 6. Задача на реальных данных: прогноз оттока абонентов

Датасет `telecom_data.csv` — данные телеком-оператора о клиентах.

### Задание 6.1. Загрузка и разведочный анализ

1. Загрузите данные
2. Выведите `shape`, `info()`, `head()`
3. Посмотрите баланс классов (`Churn`)
4. Постройте гистограммы 3–4 числовых признаков, раскрашенных по `Churn`

In [ ]:
# YOUR CODE HERE

### Задание 6.2. Предобработка

1. `Churn`: True/False → 1/0
2. `International plan`, `Voice mail plan` → бинарные
3. Удалите `State` (или закодируйте)
4. Разделите на X и y

In [ ]:
# YOUR CODE HERE

### Задание 6.3. Дерево по умолчанию

Обучите `DecisionTreeClassifier` с дефолтными параметрами. Посчитайте 5-fold CV F1 и accuracy.

In [ ]:
# YOUR CODE HERE

### Задание 6.4. GridSearchCV для дерева

Подберите `max_depth`, `min_samples_split`, `min_samples_leaf` по F1.

In [ ]:
# YOUR CODE HERE

### Задание 6.5. Влияние масштабирования

Примените `MinMaxScaler`. Сравните CV accuracy дерева с масштабированием и без.

In [ ]:
# YOUR CODE HERE

**Вопрос 6.1.** Почему масштабирование **не** влияет на дерево? А на логистическую регрессию?

*Ваш ответ:*

### Задание 6.6. Сравнение моделей

Сравните (CV F1 и accuracy на отмасштабированных данных):
1. Лучшее дерево (из GridSearch)
2. Логистическая регрессия
3. Random Forest (200 деревьев)

Выведите таблицей.

In [ ]:
# YOUR CODE HERE

### Задание 6.7. Feature Importance

Обучите Random Forest, визуализируйте `feature_importances_` горизонтальным bar chart (отсортированным).

In [ ]:
# YOUR CODE HERE

**Вопрос 6.2.** Какие признаки наиболее важны для оттока? Как интерпретировать с точки зрения бизнеса?

*Ваш ответ:*

**Вопрос 6.3.** Вы менеджер телеком-компании. Какие действия вы предпримете на основе этого анализа, чтобы снизить отток?

*Ваш ответ:*

---

## Часть 7. Итоговые задания и тесты

### Тест 7.1. Теоретические вопросы

**1.** Какой критерий разбиения используется по умолчанию в `DecisionTreeClassifier` из sklearn?
- a) Entropy
- b) Gini
- c) MSE
- d) Log-loss

**2.** Пусть в узле дерева 100 объектов: 50 класса 0 и 50 класса 1. Чему равна энтропия?
- a) 0
- b) 0.5
- c) 1
- d) 2

**3.** Пусть в узле 100 объектов: 90 класса 0 и 10 класса 1. Чему равен индекс Джини?
- a) 0.18
- b) 0.10
- c) 0.50
- d) 0.81

**4.** Какое утверждение о бэггинге **НЕВЕРНО**?
- a) Бэггинг снижает дисперсию
- b) Бэггинг не меняет смещение (в среднем)
- c) Бэггинг всегда улучшает качество по сравнению с одной моделью
- d) В бутстрап-выборку попадает ~63.2% объектов

**5.** В чём отличие Random Forest от обычного бэггинга деревьев?
- a) Используются деревья вместо линейных моделей
- b) При каждом разбиении рассматривается случайное подмножество признаков
- c) Используется бустинг вместо бутстрапа
- d) Деревья строятся без ограничения глубины

**6.** Если в ансамбле из $B$ моделей корреляция $\rho = 0.5$, а дисперсия одной модели $\sigma^2 = 1$, чему равна дисперсия ансамбля при $B \to \infty$?
- a) 0
- b) 0.25
- c) 0.5
- d) 1

**7.** Дерево решений инвариантно к:
- a) Монотонным преобразованиям признаков
- b) Масштабированию признаков
- c) И то, и другое
- d) Ни то, ни другое

### Задание 7.1. Сравнение ансамблей на make_moons

На `make_moons(n_samples=500, noise=0.3, random_state=42)` визуализируйте границы решений для:
1. Одиночное дерево (`depth=9`)
2. Бэггинг (50 деревьев, `depth=9`)
3. Random Forest (50 деревьев)
4. Логистическая регрессия

В заголовке — CV accuracy. Subplots 2×2.

In [ ]:
# YOUR CODE HERE

**Вопрос 7.1.** Сравните границы решений. Почему граница RF более гладкая? Как это связано с B-V decomposition?

*Ваш ответ:*

### Задание 7.2. Сравнение Bagging vs RF: число признаков

На `digits` постройте график CV accuracy от `max_features` (1 до $d$) для:
- Бэггинг (`BaggingClassifier` с `max_features=1.0`)
- RF (`RandomForestClassifier` с разным `max_features`)

Бэггинг — горизонтальная линия, RF — кривая.

In [ ]:
# YOUR CODE HERE

### Задание 7.3 (повышенной сложности). Взвешенное голосование

Реализуйте бэггинг с **взвешенным голосованием**: вес $b$-го дерева пропорционален его accuracy на OOB-выборке. Сравните с обычным бэггингом.

In [ ]:
# YOUR CODE HERE

---

### Резюме

В этом ноутбуке мы разобрали:

1. **Теория:** энтропия, Gini, Information Gain, формальное построение дерева
2. **Реализация с нуля:** дерево, бэггинг, Random Forest
3. **Bias-Variance Decomposition:** доказательство, экспериментальная проверка, связь с ансамблями
4. **Практика:** границы решений, кросс-валидация, подбор параметров, feature importance
5. **Реальные данные:** telecom churn prediction

**В следующем ноутбуке** — бустинги (градиентный бустинг, AdaBoost, XGBoost, LightGBM, CatBoost) и практическое мастерство CatBoost для олимпиады.